# 05 Chart And Signal Scanner - MA Cross

Visual signal inspection only. Use backtests for performance conclusions.


In [ ]:
# Cell 1 - Safe import path bootstrap

import sys
from pathlib import Path


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError('Cannot find SEN05 repo root')


ROOT = find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)


In [ ]:
# Cell 2 - Imports and notebook setup

import pandas as pd
from IPython.display import display

from core_python.strategies.ma_cross.params import SYMBOLS, TIMEFRAMES, get_symbol_params
from core_python.strategies.ma_cross.model import add_ma_cross_indicators, detect_ma_cross_signals, session_mask
from core_python.strategies.ma_cross.research_utils import (
    configure_notebook,
    plot_price_with_trades,
    plot_scan_summary,
    show_run_config,
    show_strategy_summary,
)
from core_python.strategies.ma_cross.symbol.backtest import load_backtest_data

configure_notebook()
show_strategy_summary()


In [ ]:
# Cell 3 - Scanner configuration

RUN_CONFIG = {
    'symbols': ['US30', 'US100', 'GOLD', 'DE40', 'BTCUSD'],
    'tf': 'M30',
    'max_bars': 3_000,
    'date_to': None,
    'indicator_overrides': {},
    'broker_profile': None,
}
show_run_config('MA Cross Scanner Configuration', RUN_CONFIG)


In [ ]:
# Cell 4 - Scan symbols

rows = []
frames = {}
for symbol in RUN_CONFIG['symbols']:
    cfg = get_symbol_params(symbol, broker_profile=RUN_CONFIG['broker_profile'])
    raw = load_backtest_data(cfg['symbol_id'], tf=RUN_CONFIG['tf'], max_bars=RUN_CONFIG['max_bars'], date_to=RUN_CONFIG['date_to'])
    ind = add_ma_cross_indicators(raw, RUN_CONFIG['indicator_overrides'])
    sig = detect_ma_cross_signals(ind, session_mask(ind, cfg.get('session_hours_utc', [])), sym_key=symbol)
    frames[symbol] = sig
    last_signal = sig[sig['signal'] != 0].tail(1)
    rows.append({
        'symbol': symbol,
        'last_bar': sig.index[-1],
        'last_close': sig['close'].iloc[-1],
        'last_signal_time': last_signal.index[-1] if not last_signal.empty else None,
        'last_signal': int(last_signal['signal'].iloc[-1]) if not last_signal.empty else 0,
        'ma_gap_atr': sig['ma_gap_atr'].iloc[-1],
        'signals': int((sig['signal'] != 0).sum()),
    })

scanner = pd.DataFrame(rows)
plot_scan_summary(scanner)


In [ ]:
# Cell 5 - Visual chart for selected symbol

chart_symbol = RUN_CONFIG['symbols'][0]
plot_price_with_trades(frames[chart_symbol], [], symbol=chart_symbol)
